In [1]:
import os
import time

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, from_unixtime, to_date
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    LongType,
    StructField,
    StructType,
)

# Ako si u Docker mreži, koristi container/service imena, ne localhost.
KAFKA_BOOTSTRAP_SERVERS = "urbangreen-kafka:9092"
KAFKA_TOPIC = "sensor_readings"

MINIO_ENDPOINT = "http://urbangreen-minio:9000"
MINIO_ACCESS_KEY = os.getenv("MINIO_ROOT_USER", "minioadmin")
MINIO_SECRET_KEY = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin123")
MINIO_BUCKET = os.getenv("MINIO_STAGING_BUCKET", "staging")

spark = (
    SparkSession.builder
    .appName("sensor-readings-stream-notebook-test")
    .master("spark://urbangreen-spark-master:7077")
    .config("spark.driver.host", "urbangreen-jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.port", "7078")
    .config("spark.blockManager.port", "7079")
    .config("spark.executor.memory", "512m")
    .config("spark.executor.cores", "1")
    .config("spark.cores.max", "2")

    # Kafka connector za Spark 4.x
    .config(
        "spark.jars.packages",
        ",".join([
            "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.2",
            "org.apache.hadoop:hadoop-aws:3.4.1",
        ])
    )

    # MinIO / S3A config
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("MinIO endpoint:", MINIO_ENDPOINT)
print("Bucket:", MINIO_BUCKET)

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jovyan/.ivy2.5.2/cache
The jars for the packages stored in: /home/jovyan/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d38a5a92-cdb9-45e4-b4c4-023584ed2552;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.0.2 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.0.2 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found org.slf4j#slf4j-api;2.0.16 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.1 in central
	found org.apache.hadoop#hadoop-client-api;3.4.1 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala

Spark version: 4.0.2
MinIO endpoint: http://urbangreen-minio:9000
Bucket: staging


In [2]:
df = spark.range(1000)
print(df.count())

1000


In [3]:
spark.range(10).selectExpr("CAST(id AS STRING) AS id").show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
|  5|
|  6|
|  7|
|  8|
|  9|
+---+



In [4]:
SENSOR_SCHEMA = StructType(
    [
        StructField("farm_sensor_id", IntegerType(), nullable=False),
        StructField("farm_id", IntegerType(), nullable=False),
        StructField("sensor_type_id", IntegerType(), nullable=False),
        StructField("value", DoubleType(), nullable=False),
        StructField("timestamp", LongType(), nullable=False),
    ]
)

In [5]:
kafka_batch = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .option("endingOffsets", "latest")
    .load()
)

raw_messages = kafka_batch.selectExpr(
    "CAST(key AS STRING) AS key",
    "CAST(value AS STRING) AS value",
    "topic",
    "partition",
    "offset",
    "timestamp AS kafka_timestamp"
)

raw_messages.show(truncate=False)

+---+----------------------------------------------------------------------------------------------------+---------------+---------+------+-----------------------+
|key|value                                                                                               |topic          |partition|offset|kafka_timestamp        |
+---+----------------------------------------------------------------------------------------------------+---------------+---------+------+-----------------------+
|1  |{"farm_sensor_id": 1, "farm_id": 1, "sensor_type_id": 1, "value": 21.279, "timestamp": 1752065112}  |sensor_readings|0        |0     |2026-07-09 12:45:12.163|
|5  |{"farm_sensor_id": 5, "farm_id": 1, "sensor_type_id": 5, "value": 59.913, "timestamp": 1752065112}  |sensor_readings|0        |1     |2026-07-09 12:45:12.163|
|7  |{"farm_sensor_id": 7, "farm_id": 2, "sensor_type_id": 1, "value": 21.144, "timestamp": 1752065112}  |sensor_readings|0        |2     |2026-07-09 12:45:12.164|
|8  |{"farm_sens

In [6]:
parsed_batch = (
    raw_messages
    .select(
        "key",
        "topic",
        "partition",
        "offset",
        "kafka_timestamp",
        from_json(col("value"), SENSOR_SCHEMA).alias("payload")
    )
    .select(
        "key",
        "topic",
        "partition",
        "offset",
        "kafka_timestamp",
        "payload.*"
    )
    .withColumn("event_date", to_date(from_unixtime(col("timestamp"))))
)

parsed_batch.show(20, truncate=False)
parsed_batch.printSchema()

+---+---------------+---------+------+-----------------------+--------------+-------+--------------+-------+----------+----------+
|key|topic          |partition|offset|kafka_timestamp        |farm_sensor_id|farm_id|sensor_type_id|value  |timestamp |event_date|
+---+---------------+---------+------+-----------------------+--------------+-------+--------------+-------+----------+----------+
|1  |sensor_readings|0        |0     |2026-07-09 12:45:12.163|1             |1      |1             |21.279 |1752065112|2025-07-09|
|5  |sensor_readings|0        |1     |2026-07-09 12:45:12.163|5             |1      |5             |59.913 |1752065112|2025-07-09|
|7  |sensor_readings|0        |2     |2026-07-09 12:45:12.164|7             |2      |1             |21.144 |1752065112|2025-07-09|
|8  |sensor_readings|0        |3     |2026-07-09 12:45:12.164|8             |2      |2             |58.379 |1752065112|2025-07-09|
|11 |sensor_readings|0        |4     |2026-07-09 12:45:12.164|11            |2     

In [7]:
from pyspark.sql.functions import count, when

null_check = parsed_batch.select(
    count(when(col("farm_sensor_id").isNull(), True)).alias("null_farm_sensor_id"),
    count(when(col("farm_id").isNull(), True)).alias("null_farm_id"),
    count(when(col("sensor_type_id").isNull(), True)).alias("null_sensor_type_id"),
    count(when(col("value").isNull(), True)).alias("null_value"),
    count(when(col("timestamp").isNull(), True)).alias("null_timestamp"),
    count(when(col("event_date").isNull(), True)).alias("null_event_date"),
)

null_check.show()

+-------------------+------------+-------------------+----------+--------------+---------------+
|null_farm_sensor_id|null_farm_id|null_sensor_type_id|null_value|null_timestamp|null_event_date|
+-------------------+------------+-------------------+----------+--------------+---------------+
|                  0|           0|                  0|         0|             0|              0|
+-------------------+------------+-------------------+----------+--------------+---------------+



In [8]:
import os

print("MINIO_ROOT_USER:", os.getenv("MINIO_ROOT_USER"))
print("MINIO_ROOT_PASSWORD exists:", os.getenv("MINIO_ROOT_PASSWORD") is not None)
print("MINIO_STAGING_BUCKET:", os.getenv("MINIO_STAGING_BUCKET"))

MINIO_ROOT_USER: None
MINIO_ROOT_PASSWORD exists: False
MINIO_STAGING_BUCKET: None


In [9]:
test_output_path = f"s3a://{MINIO_BUCKET}/raw/kafka/{KAFKA_TOPIC}_notebook_batch_test/"

(
    parsed_batch
    .select(
        "farm_sensor_id",
        "farm_id",
        "sensor_type_id",
        "value",
        "timestamp",
        "event_date",
    )
    .write
    .mode("overwrite")
    .partitionBy("event_date")
    .parquet(test_output_path)
)

print("Written to:", test_output_path)

Py4JJavaError: An error occurred while calling o172.parquet.
: java.lang.RuntimeException: java.lang.ClassNotFoundException: Class org.apache.hadoop.fs.s3a.S3AFileSystem not found
	at org.apache.hadoop.conf.Configuration.getClass(Configuration.java:2737)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3569)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3612)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:172)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3716)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3667)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:557)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:366)
	at org.apache.spark.sql.execution.datasources.DataSource.makeQualified(DataSource.scala:125)
	at org.apache.spark.sql.execution.datasources.DataSource.planForWritingFileFormat(DataSource.scala:468)
	at org.apache.spark.sql.execution.datasources.DataSource.plan

In [ ]:
stream_output_path = f"s3a://{MINIO_BUCKET}/raw/kafka/{KAFKA_TOPIC}_notebook_stream_test/"
stream_checkpoint_path = f"s3a://{MINIO_BUCKET}/_checkpoints/kafka/{KAFKA_TOPIC}_notebook_stream_test/"

In [ ]:
kafka_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .option("failOnDataLoss", "false")
    .load()
)

parsed_stream = (
    kafka_stream
    .select(
        from_json(col("value").cast("string"), SENSOR_SCHEMA).alias("payload")
    )
    .select("payload.*")
    .withColumn("event_date", to_date(from_unixtime(col("timestamp"))))
)

query = (
    parsed_stream
    .writeStream
    .format("parquet")
    .option("path", stream_output_path)
    .option("checkpointLocation", stream_checkpoint_path)
    .outputMode("append")
    .partitionBy("event_date")
    .trigger(processingTime="10 seconds")
    .start()
)

print("Streaming started")
print("Output:", stream_output_path)
print("Checkpoint:", stream_checkpoint_path)

In [ ]:
query.status

In [ ]:
stream_result = spark.read.parquet(stream_output_path)

stream_result.show(20, truncate=False)
stream_result.groupBy("event_date").count().show(truncate=False)
stream_result.printSchema()